In [ ]:
import os, glob, pickle
import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA


# ---------------------------- CONFIG ---------------------------------
BASE    = '/'
EMB_DIR = os.path.join(BASE, 'meanpooling')
OUT_DIR = os.path.join(BASE, 'PCA')

MODALITIES = ['http', 'file', 'email']
USERS = [

 ]

EMB_DIM_EXPECTED = 384
N_COMPONENTS     = 32
EXPECTED_KEYS    = {'user', 'modality', 'dates', 'embeddings'}

os.makedirs(OUT_DIR, exist_ok=True)
for m in MODALITIES:
    os.makedirs(os.path.join(OUT_DIR, m), exist_ok=True)

_ERRORS, _WARNINGS = [], []
def err(m):  _ERRORS.append(m);   print(f'[ERROR]   {m}')
def warn(m): _WARNINGS.append(m); print(f'[WARNING] {m}')
def ok(m):   print(f'[OK]      {m}')
def section(t): print('\n' + '=' * 72 + f'\n{t}\n' + '=' * 72)
def stop_if_errors(ctx):
    if _ERRORS:
        raise RuntimeError(f'{ctx}: ERROR {len(_ERRORS)}건 발생. 위 로그를 먼저 해결해야 한다.\n'
                           + '\n'.join(f'  - {e}' for e in _ERRORS))

# ==================== 1. 파일 구조 검증 ==============================
section('1. PATH / FILE STRUCTURE VALIDATION')

if not os.path.isdir(BASE):
    raise FileNotFoundError(f'BASE 경로가 없다: {BASE}')
if not os.path.isdir(EMB_DIR):
    raise FileNotFoundError(f'meanpooling 디렉터리가 없다: {EMB_DIR}')
ok(f'BASE    = {BASE}')
ok(f'EMB_DIR = {EMB_DIR}')
ok(f'OUT_DIR = {OUT_DIR}')
print('\nmeanpooling 하위:', sorted(os.listdir(EMB_DIR)))

pkl_paths = {}
for m in MODALITIES:
    mdir = os.path.join(EMB_DIR, m)
    if not os.path.isdir(mdir):
        err(f'modality 디렉터리 없음: {mdir}')
        continue

    n_pkl = len(glob.glob(os.path.join(mdir, '*.pkl')))
    print(f'  {m:6s} : pkl {n_pkl}개')
    if n_pkl != len(USERS):
        warn(f'{m}: 디렉터리의 pkl 수가 {n_pkl}개로 USERS({len(USERS)})와 다르다.')

    for u in USERS:
        p = os.path.join(mdir, f'{m}_{u}_meanpooled.pkl')
        if os.path.isfile(p):
            pkl_paths[(m, u)] = p
        else:
            err(f'파일 없음: {p}')

stop_if_errors('file inventory')
ok(f'{len(USERS)} users x {len(MODALITIES)} modalities = {len(pkl_paths)} pkl 전부 확인')

# ==================== 2. 로딩 + 스키마 검증 ==========================
section('2. LOAD MEAN-POOLED EMBEDDINGS')

emb_store, schema_rows = {}, []

for (m, u), path in pkl_paths.items():
    try:
        with open(path, 'rb') as f:
            obj = pickle.load(f)
    except Exception as e:
        err(f'{m}/{u}: pkl 로딩 실패 -> {type(e).__name__}: {e}'); continue

    if not isinstance(obj, dict):
        err(f'{m}/{u}: dict 가 아니라 {type(obj)}'); continue
    miss = EXPECTED_KEYS - set(obj.keys())
    if miss:
        err(f'{m}/{u}: 누락 key {miss} (실제 key: {list(obj.keys())})'); continue
    if obj['user'] != u:
        err(f'{m}/{u}: pkl 내부 user="{obj["user"]}" 가 파일명과 불일치')
    if obj['modality'] != m:
        err(f'{m}/{u}: pkl 내부 modality="{obj["modality"]}" 가 디렉터리와 불일치')

    dates = [str(pd.to_datetime(d).date()) for d in list(obj['dates'])]
    emb   = np.asarray(obj['embeddings'], dtype=np.float32)

    if emb.ndim != 2:
        err(f'{m}/{u}: embeddings 가 2D 가 아니다 -> {emb.shape}'); continue
    if emb.shape[0] != len(dates):
        err(f'{m}/{u}: dates({len(dates)}) vs embeddings({emb.shape[0]}) 길이 불일치'); continue
    if emb.shape[1] != EMB_DIM_EXPECTED:
        err(f'{m}/{u}: 차원 {emb.shape[1]} (기대값 {EMB_DIM_EXPECTED})'); continue
    if not np.isfinite(emb).all():
        err(f'{m}/{u}: NaN/Inf {int((~np.isfinite(emb)).sum())}개'); continue
    if len(set(dates)) != len(dates):
        err(f'{m}/{u}: dates 중복 {len(dates) - len(set(dates))}건'); continue
    if dates != sorted(dates):
        warn(f'{m}/{u}: dates 가 시간순이 아니다. 정렬 후 사용한다.')
        order = np.argsort(dates)
        dates = [dates[i] for i in order]
        emb   = emb[order]

    norms = np.linalg.norm(emb, axis=1)
    schema_rows.append({'modality': m, 'user': u, 'n_days': len(dates),
                        'dim': int(emb.shape[1]),
                        'date_min': min(dates), 'date_max': max(dates),
                        'norm_mean': round(float(norms.mean()), 4),
                        'zero_rows': int((norms < 1e-8).sum())})
    emb_store[(m, u)] = {'dates': dates, 'embeddings': emb, 'src': path}

schema = pd.DataFrame(schema_rows)

print('\nmodality 별 로딩 요약:')
print(schema.groupby('modality').agg(
    users=('user', 'nunique'), total_rows=('n_days', 'sum'),
    days_min=('n_days', 'min'), days_max=('n_days', 'max'),
    dim=('dim', 'max'), zero_rows=('zero_rows', 'sum')).to_string())

print('\n사용자별 row count (PCA 전):')
print(schema.pivot(index='user', columns='modality', values='n_days')
            .reindex(USERS)[MODALITIES].to_string())

if set(emb_store.keys()) != {(m, u) for m in MODALITIES for u in USERS}:
    err(f'로딩된 (modality,user) {len(emb_store)} != 기대 {len(MODALITIES)*len(USERS)}')
if (schema['zero_rows'] > 0).any():
    warn('노름이 0 인 임베딩 행이 있다. 빈 텍스트가 그대로 pooling 되었을 수 있다.')
ok(f'unique user count = {schema["user"].nunique()} (기대 {len(USERS)})')
print('positive label count : N/A  (이 단계는 embedding-only, label CSV 미사용)')
stop_if_errors('embedding load')

# ==================== 3. modality 별 PCA 384 -> 32 ===================
section(f'3. PCA {EMB_DIM_EXPECTED}D -> {N_COMPONENTS}D  (modality별 축 1개, 사용자별 저장)')

count_rows, saved_files = [], []

for m in MODALITIES:
    print(f'\n--- {m} ---')

    # (a) 축을 구하기 위해서만 20명을 쌓는다. 출력은 다시 사용자별로 분리한다.
    parts  = [emb_store[(m, u)]['embeddings'] for u in USERS]
    n_each = [len(emb_store[(m, u)]['dates']) for u in USERS]
    X      = np.concatenate(parts, axis=0).astype(np.float64)
    n_rows, n_dim = X.shape

    print(f'  fit 표본   : {n_rows:,} rows   user {len(USERS)}명   input dim {n_dim}')
    if n_rows != sum(n_each):
        err(f'{m}: stacked rows {n_rows} != sum(per-user) {sum(n_each)}'); continue
    if n_dim != EMB_DIM_EXPECTED:
        err(f'{m}: input dim {n_dim} != {EMB_DIM_EXPECTED}'); continue
    if n_rows < N_COMPONENTS:
        err(f'{m}: 표본 {n_rows}행 < n_components {N_COMPONENTS}. PCA 불가.'); continue

    mu = X.mean(axis=0)
    print(f'  ||global mean vector|| = {np.linalg.norm(mu):.4f}')
    print(f'  mean deviation norm    = {np.linalg.norm(X - mu, axis=1).mean():.4f}')

    pca = PCA(n_components=N_COMPONENTS, svd_solver='full', random_state=42)
    pca.fit(X)

    cs = np.cumsum(pca.explained_variance_ratio_)
    print(f'  explained variance @{N_COMPONENTS} : {cs[-1]*100:.2f}%   (PC1 {pca.explained_variance_ratio_[0]*100:.2f}%)')
    for k in (2, 4, 8, 16, 32):
        if k <= N_COMPONENTS:
            print(f'      k={k:2d} -> {cs[k-1]*100:6.2f}%')

    # (b) 사용자별로 transform 하고 사용자별 파일로 저장한다.
    for u in USERS:
        rec   = emb_store[(m, u)]
        n_src = len(rec['dates'])
        z_u   = pca.transform(rec['embeddings'].astype(np.float64)).astype(np.float32)

        if z_u.shape != (n_src, N_COMPONENTS):
            err(f'{m}/{u}: transform shape {z_u.shape} != ({n_src}, {N_COMPONENTS})'); continue
        if not np.isfinite(z_u).all():
            err(f'{m}/{u}: PCA 출력에 NaN/Inf'); continue

        out_path = os.path.join(OUT_DIR, m, f'{m}_{u}_pca{N_COMPONENTS}.pt')
        torch.save({
            'user': u,
            'modality': m,
            'dates': rec['dates'],
            'embeddings': torch.from_numpy(np.ascontiguousarray(z_u)),  # (T, 32)
            'input_dim': EMB_DIM_EXPECTED,
            'n_components': N_COMPONENTS,
            'source_pkl': rec['src'],
            'explained_variance_ratio_sum': float(cs[-1]),
        }, out_path)
        saved_files.append(out_path)

        count_rows.append({'modality': m, 'user': u,
                           'n_days_before': n_src, 'n_days_after': int(z_u.shape[0]),
                           'dim_before': EMB_DIM_EXPECTED, 'dim_after': int(z_u.shape[1]),
                           'row_count_match': n_src == int(z_u.shape[0])})

    print(f'  저장: {os.path.join(OUT_DIR, m)}/{m}_{{user}}_pca{N_COMPONENTS}.pt  '
          f'({sum(1 for r in count_rows if r["modality"] == m)}개)')

stop_if_errors('PCA fit/transform')

# ==================== 4. 축소 후 검증 ================================
section('4. POST-PCA VALIDATION')

counts = pd.DataFrame(count_rows)

if len(saved_files) != len(MODALITIES) * len(USERS):
    err(f'저장 파일 {len(saved_files)}개 != 기대 {len(MODALITIES) * len(USERS)}개')
else:
    ok(f'저장 파일 {len(saved_files)}개 (입력 pkl 60개와 1:1 대응)')

if len(counts[~counts['row_count_match']]):
    err('row count 불일치 발생')
    print(counts[~counts['row_count_match']].to_string(index=False))
else:
    ok('모든 (modality, user) 에서 PCA 전/후 row count 동일')

if set(counts['user']) != set(USERS):
    err('검증 테이블의 user 집합이 USERS 와 다르다')
else:
    ok(f'user count = {counts["user"].nunique()}')

if (counts['dim_after'] != N_COMPONENTS).any():
    err(f'출력 차원이 {N_COMPONENTS} 가 아닌 행이 있다')
else:
    ok(f'모든 출력 차원 = {N_COMPONENTS}')

print('\nmodality 별 총 row count:')
print(counts.groupby('modality').agg(files=('user', 'nunique'),
                                     rows_before=('n_days_before', 'sum'),
                                     rows_after=('n_days_after', 'sum')).to_string())

print('\n사용자별 row count (PCA 후):')
print(counts.pivot(index='user', columns='modality', values='n_days_after')
            .reindex(USERS)[MODALITIES].to_string())

print('\n저장 파일 재로딩 검증:')
for m in MODALITIES:
    n_ok = 0
    for u in USERS:
        p = os.path.join(OUT_DIR, m, f'{m}_{u}_pca{N_COMPONENTS}.pt')
        if not os.path.isfile(p):
            err(f'저장 파일 없음: {p}'); continue
        try:
            obj = torch.load(p, map_location='cpu', weights_only=False)
        except TypeError:
            obj = torch.load(p, map_location='cpu')

        n_src = len(emb_store[(m, u)]['dates'])
        if obj['user'] != u or obj['modality'] != m:
            err(f'{m}/{u}: 저장된 user/modality 불일치 -> {obj["user"]}/{obj["modality"]}')
        if tuple(obj['embeddings'].shape) != (n_src, N_COMPONENTS):
            err(f'{m}/{u}: reload shape {tuple(obj["embeddings"].shape)} != ({n_src}, {N_COMPONENTS})')
        if obj['dates'] != emb_store[(m, u)]['dates']:
            err(f'{m}/{u}: reload dates 가 원본과 다르다')
        n_ok += 1
    ok(f'{m}: {n_ok}/{len(USERS)} 파일 reload 검증 완료')

# ==================== 5. SUMMARY =====================================
section('5. SUMMARY')

print(f'출력 디렉터리: {OUT_DIR}')
for m in MODALITIES:
    d = os.path.join(OUT_DIR, m)
    fs = sorted(os.listdir(d)) if os.path.isdir(d) else []
    print(f'  {m}/  ({len(fs)} files)')
    for fn in fs[:3]:
        print(f'    - {fn}')
    if len(fs) > 3:
        print(f'    ... 외 {len(fs) - 3}개')

print(f'\nERROR   : {len(_ERRORS)}건')
for e in _ERRORS:   print('  -', e)
print(f'WARNING : {len(_WARNINGS)}건')
for w in _WARNINGS: print('  -', w)

if _ERRORS:
    print('\n>>> ERROR 가 있다. 다음 단계로 넘어가기 전에 해결해야 한다.')
else:
    print('\n>>> ERROR 없음. 384D -> 32D 축소 완료 (60 files).')
    print('    다음 단계(아직 구현 안 함): PCA 32D 를 daily_features 와 (user, date) 로 join')